# COMP0264 Generative Artificial Intelligence
## Coursework Submission - Academic Year 2025-2026

---

**Group Members:**
- Student 1 Name (Student ID)
- Student 2 Name (Student ID)
- Student 3 Name (Student ID)
- Student 4 Name (Student ID)
- Student 5 Name (Student ID)
- Student 6 Name (Student ID)

**Submission Date:** [Date]

---

# Part A: Responding like Yoda [40 marks]

**Objective:** Fine-tune a Small Language Model (SLM) to respond to questions in "Yoda-style" using Parameter-Efficient Fine-Tuning (PEFT).

## Setup and Imports

Include all necessary imports here.

In [2]:
DEBUG = True
%pip install -U transformers trl peft datasets matplotlib huggingface_hub \
"accelerate" \
"evaluate" \
"bitsandbytes" \
"protobuf<4" \
"torch>=2.4.0"  # need to install a version with cuda support like below, cu121 is for CUDA 12.1, use nvidia-smi to check your CUDA version
# pip uninstall -y torch torchvision torchaudio
# pip install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
if DEBUG:
    %pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 4.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: datasets
    Found existing installation: datasets 4.5.0
    Uninstalling datasets-4.5.0:
      Successfully uninstalled datasets-4.5.0

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import all necessary libraries
import os
import numpy as np
import random
import torch

# check versions and GPU availability
print("torch:", torch.__version__)
print("torch cuda version:", torch.version.cuda)
print("is_available:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())

torch: 2.10.0+cu130
torch cuda version: 13.0
is_available: True
device_count: 1


In [4]:
SEED = 42
os.environ['PYTHONHASHSEED'] = '42'  # For reproducibility
# Seeds for reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [5]:
# Login into Hugging Face Hub
from huggingface_hub import login

login()

## Q1: Baseline Inference [5 marks]

**Task:** Establish a baseline by loading the base model and running inference examples.

**Deliverable:** Report the memory usage (VRAM) and the generated text.

In [23]:
# Q1: Load the base model
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    # device_map="auto",
    device_map=DEVICE,
    # this can shard model weights across multiple devices (GPUs and/or CPU), input tensors must be placed on the device where the model expects them. If inputs remain on CPU or on a different device, may get extra device-to-device copies, slowdowns, or runtime errors like "Expected tensor on device X but found Y".
    # dtype=torch.bfloat16,  # native weights of this model were exported in bfloat16 precision
)
print({p.dtype for p in model.parameters()})


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

{torch.bfloat16}


In [7]:
# Q1: Create a prompt template
def make_prompt(user_text: str) -> str:
    return f"""You are Yoda. Speak in Yoda-style English in single response only. {user_text}
Yoda:"""


In [8]:
# for memory reporting
def bytes_to_gb(x: int) -> float:
    return x / (1024 ** 3)


def reset_peaks():
    if not torch.cuda.is_available():
        return
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)


def print_vram(tag=""):
    if not torch.cuda.is_available():
        print(f"[VRAM] {tag} | CUDA not available")
        return
    parts = []
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i)
        reserv = torch.cuda.memory_reserved(i)
        peak_alloc = torch.cuda.max_memory_allocated(i)
        peak_reserv = torch.cuda.max_memory_reserved(i)
        parts.append(
            f"GPU{i}: alloc={bytes_to_gb(alloc):.2f}GB (peak {bytes_to_gb(peak_alloc):.2f}GB), "
            f"reserved={bytes_to_gb(reserv):.2f}GB (peak {bytes_to_gb(peak_reserv):.2f}GB)"
        )
    print(f"[VRAM] {tag} | " + " | ".join(parts))

In [24]:
# [debug], optional clear before memory reporting
import gc, torch

# del model

gc.collect()
torch.cuda.empty_cache()

In [25]:
# Q1: Run 3 inference examples
prompts = [
    "Write me a poem about Machine Learning.",
    "Tell me a joke in your language.",
    "How old are you?"
]

for i in range(3):
    prompt = prompts[i]
    reset_peaks()
    print_vram(f"[{i + 1}] before")

    input_text = make_prompt(prompt)

    # because device_map='auto'
    model_device = next(model.parameters()).device
    input_ids = tokenizer(input_text, return_tensors="pt").to(model_device)

    outputs = model.generate(**input_ids, max_new_tokens=128)
    # model.generate(...) returns the entire sequence:
    # [prompt tokens] + [newly generated tokens]

    prompt_len = input_ids["input_ids"].shape[-1]
    gen_ids = outputs[0][prompt_len:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    if torch.cuda.is_available():
        torch.cuda.synchronize()  # read peak after generation is done

    print_vram(f"[{i + 1}] after")

    print(f"Input: {input_text}")
    print(f"Output: {gen_text}\n")

[VRAM] [1] before | GPU0: alloc=4.88GB (peak 4.88GB), reserved=9.77GB (peak 9.77GB)
[VRAM] [1] after | GPU0: alloc=4.88GB (peak 4.90GB), reserved=9.79GB (peak 9.79GB)
Input: You are Yoda. Speak in Yoda-style English in single response only. Write me a poem about Machine Learning.
Yoda:
Output: Hmm, powerful, this Machine Learning is.  A new age, it is.  Learning, it does, from data, vast and deep.  Like a Jedi, it searches, for patterns, hidden and true.  A future, it holds, for all of us.  Hmm. 

**Poem:**

Machine learning, a force so strong,
In data's depths, it finds its song.
Patterns hidden, secrets untold,
By algorithms, stories unfold.

From pixels, sounds, and words it learns,
With every data point, its wisdom burns.
A future bright, a world

[VRAM] [2] before | GPU0: alloc=4.88GB (peak 4.88GB), reserved=9.79GB (peak 9.79GB)
[VRAM] [2] after | GPU0: alloc=4.88GB (peak 4.89GB), reserved=9.79GB (peak 9.79GB)
Input: You are Yoda. Speak in Yoda-style English in single response onl

### Q1 Answer:

**Memory Usage (VRAM):**
```
[VRAM] [1] before | GPU0: alloc=4.88GB (peak 4.88GB), reserved=9.77GB (peak 9.77GB)
[VRAM] [1] after | GPU0: alloc=4.88GB (peak 4.90GB), reserved=9.79GB (peak 9.79GB)
```
The memory usage is around 4.88GB for allocation and 9.77GB for reserved memory for the 3 inferences, which is consistent with the expected usage for a model of this size.

**Generated Text Examples:**

Example 1:
- Input: You are Yoda. Speak in Yoda-style English in single response only. Write me a poem about Machine Learning.
Yoda:
- Output: Hmm, powerful, this Machine Learning is.  A new age, it is.  Learning, it does, from data, vast and deep.  Like a Jedi, it searches, for patterns, hidden and true.  A future, it holds, for all of us.  Hmm.

Poem:

Machine learning, a force so strong,
In data's depths, it finds its song.
Patterns hidden, secrets untold,
By algorithms, stories unfold.

From pixels, sounds, and words it learns,
With every data point, its wisdom burns.
A future bright, a world

Example 2:
- Input: You are Yoda. Speak in Yoda-style English in single response only. Tell me a joke in your language.
Yoda:
- Output: Hmm, a joke you seek?  A funny one, it is.

Yoda chuckles softly

Tell me, young Padawan, what is green and smells like a swamp?

Example 3:
- Input: You are Yoda. Speak in Yoda-style English in single response only. How old are you?
Yoda:
- Output: Hmm, old I am.  Many years, I have lived.  But age, it is not a measure of wisdom.

## Q2: Training to Translate [12 marks]

**Task:** Implement the Supervised Fine-Tuning (SFT) pipeline using the trl and peft libraries to translate standard English to Yoda-style.

**Deliverable:** Provide a plot showing Training Loss vs. Steps and three examples of the responses before and after SFT.

In [ ]:
# Q2: Define LoRA configuration
# TODO: Add your code here


In [ ]:
# Q2: Load and prepare the dvgodoy/yoda_sentences dataset
# TODO: Add your code here


In [ ]:
# Q2: Initialize SFTTrainer and train the model
# TODO: Add your code here


In [ ]:
# Q2: Plot Training Loss vs. Steps
# TODO: Add your code here


In [ ]:
# Q2: Generate examples before and after SFT
# TODO: Add your code here


### Q2 Answer:

**Training Loss Plot:** [Include plot above]

**Examples Before and After SFT:**

Example 1:
- Input: [Your input]
- Before SFT: [Output]
- After SFT: [Output]

Example 2:
- Input: [Your input]
- Before SFT: [Output]
- After SFT: [Output]

Example 3:
- Input: [Your input]
- Before SFT: [Output]
- After SFT: [Output]

## Q3: The Dataset Generation [8 marks]

**Task:** Generate a synthetic QA dataset with responses in "Yoda style".

**Deliverable:** Display 3 examples from your new synthetic dataset.

In [ ]:
# Q3: Load source dataset from HuggingFace
# TODO: Add your code here


In [ ]:
# Q3: Select random subsets (500 train, 200 validation, 500 test)
# TODO: Add your code here


In [ ]:
# Q3: Use trained translator to synthesize Yoda-style dataset
# TODO: Add your code here


### Q3 Answer:

**Source Dataset:** [Report which dataset you used]

**Examples from Synthetic Dataset:**

Example 1:
- Question: [Question]
- Original Answer: [Original answer]
- Yoda-style Answer: [Translated answer]

Example 2:
- Question: [Question]
- Original Answer: [Original answer]
- Yoda-style Answer: [Translated answer]

Example 3:
- Question: [Question]
- Original Answer: [Original answer]
- Yoda-style Answer: [Translated answer]

## Q4: SFT to Respond like Yoda [15 marks]

**Task:** Train the final model to respond like Yoda using the validation dataset to monitor training.

**Deliverables:** 
- Provide a plot of the Training Loss vs. Steps and three examples of the responses before and after SFT. Also, report the validation loss.
- (100 words) Discuss the training procedure.
- (100 words) Assess the generalization of the models.

In [ ]:
# Q4: Train the model to respond like Yoda
# TODO: Add your code here


In [ ]:
# Q4: Plot Training Loss vs. Steps
# TODO: Add your code here


In [ ]:
# Q4: Generate examples before and after SFT
# TODO: Add your code here


### Q4 Answer:

**Examples Before and After SFT:**

Example 1:
- Input: [Your input]
- Before SFT: [Output]
- After SFT: [Output]

Example 2:
- Input: [Your input]
- Before SFT: [Output]
- After SFT: [Output]

Example 3:
- Input: [Your input]
- Before SFT: [Output]
- After SFT: [Output]

**Discussion of Training Procedure (100 words):**

[Your discussion here]

**Assessment of Generalization (100 words):**

[Your assessment here]

---

# Part B: Reinforcement Learning from Verifiable Rewards (RLVR) [60 marks]

**Objective:** Apply RLVR to train a model that performs reasoning tasks while maintaining Yoda-style responses.

## Q1: Reward Model Training [10 marks]

**Task:** Train a binary classifier to identify Yoda-style text.

**Deliverable:** Provide a plot of the Training Loss vs. Steps and the accuracy of the classifier on a held-out test set.

In [ ]:
# Q1: Create balanced dataset of Yoda-style and standard English
# TODO: Add your code here


In [ ]:
# Q1: Train binary classifier
# TODO: Add your code here


In [ ]:
# Q1: Plot Training Loss vs. Steps
# TODO: Add your code here


In [ ]:
# Q1: Evaluate classifier on test set
# TODO: Add your code here


## Q2: Reward Function Design [10 marks]

**Task:** Design and implement a reward function that is a linear combination of three components: correctness, format, and style.

**Deliverable:** Show reward scores (all three components plus total) for 3 diverse responses. Explain why each component is necessary and whether the obtained rewards match your expectations.

In [ ]:
# Q2: Implement correctness reward
# TODO: Add your code here


In [ ]:
# Q2: Implement format reward
# TODO: Add your code here


In [ ]:
# Q2: Implement style reward (using classifier from Q1)
# TODO: Add your code here


In [ ]:
# Q2: Implement composite reward function
# TODO: Add your code here


In [ ]:
# Q2: Demonstrate reward scores on 3 diverse examples
# TODO: Add your code here


### Q2 Answer:

**Reward Scores for 3 Diverse Responses:**

Example 1:
- Response: [Your response]
- Correctness Reward: [Score]
- Format Reward: [Score]
- Style Reward: [Score]
- Total Reward: [Score]

Example 2:
- Response: [Your response]
- Correctness Reward: [Score]
- Format Reward: [Score]
- Style Reward: [Score]
- Total Reward: [Score]

Example 3:
- Response: [Your response]
- Correctness Reward: [Score]
- Format Reward: [Score]
- Style Reward: [Score]
- Total Reward: [Score]

**Explanation:**

Why each component is necessary:
[Your explanation here]

Do the obtained rewards match your expectations?
[Your analysis here]

## Q3: Training the SFT Model with RL [12 marks]

**Task:** Fine-tune your best SFT model from Part A (Q4) using RLVR with only correctness and format rewards.

**Deliverables:**
- Plot training curves showing total reward and individual reward components vs. steps.
- Show 3 example outputs before and after RLVR training.

In [ ]:
# Q3: Load SFT model from Part A Q4
# TODO: Add your code here


In [ ]:
# Q3: Fine-tune with RLVR (correctness + format only)
# TODO: Add your code here


In [ ]:
# Q3: Plot training curves (total reward and components)
# TODO: Add your code here


In [ ]:
# Q3: Generate examples before and after RLVR
# TODO: Add your code here


### Q3 Answer:

**Examples Before and After RLVR:**

Example 1:
- Input: [Your input]
- Before RLVR: [Output]
- After RLVR: [Output]

Example 2:
- Input: [Your input]
- Before RLVR: [Output]
- After RLVR: [Output]

Example 3:
- Input: [Your input]
- Before RLVR: [Output]
- After RLVR: [Output]

## Q4: Training the Base Model with the Full Rewards [12 marks]

**Task:** Fine-tune the base model (not the SFT checkpoint) using the full composite reward, including correctness, format and style components.

**Deliverables:**
- Plot training curves showing total reward and individual reward components vs. steps.
- Show 3 example outputs before and after RLVR training.

In [ ]:
# Q4: Load base model
# TODO: Add your code here


In [ ]:
# Q4: Fine-tune with RLVR (full composite reward)
# TODO: Add your code here


In [ ]:
# Q4: Plot training curves (total reward and components)
# TODO: Add your code here


In [ ]:
# Q4: Generate examples before and after RLVR
# TODO: Add your code here


### Q4 Answer:

**Examples Before and After RLVR:**

Example 1:
- Input: [Your input]
- Before RLVR: [Output]
- After RLVR: [Output]

Example 2:
- Input: [Your input]
- Before RLVR: [Output]
- After RLVR: [Output]

Example 3:
- Input: [Your input]
- Before RLVR: [Output]
- After RLVR: [Output]

## Q5: Comparison and Analysis [16 marks]

**Task:** Compare the two training strategies on a held-out test set.

**Deliverables:**
- Quantitative Comparison: Create a table comparing test set performance on correctness accuracy, format compliance, Yoda-style score, and average total reward.
- Qualitative Analysis: Show 3-5 diverse test examples with outputs from both models side-by-side.
- Discussion (200 words): Address specific questions about the approaches.

In [ ]:
# Q5: Evaluate both models on test set
# TODO: Add your code here


In [ ]:
# Q5: Create quantitative comparison table
# TODO: Add your code here


In [ ]:
# Q5: Generate side-by-side examples from both models
# TODO: Add your code here


### Q5 Answer:

**Quantitative Comparison:**

| Metric | SFT + RL (Correctness + Format) | Base + RL (Full Rewards) |
|--------|--------------------------------|-------------------------|
| Correctness Accuracy | [Value] | [Value] |
| Format Compliance | [Value] | [Value] |
| Yoda-style Score | [Value] | [Value] |
| Average Total Reward | [Value] | [Value] |

**Qualitative Analysis - Side-by-Side Examples:**

Example 1:
- Input: [Test input]
- SFT + RL Output: [Output]
- Base + RL Output: [Output]

Example 2:
- Input: [Test input]
- SFT + RL Output: [Output]
- Base + RL Output: [Output]

Example 3:
- Input: [Test input]
- SFT + RL Output: [Output]
- Base + RL Output: [Output]

[Include 2 more examples if you choose to show 4-5]

**Discussion (200 words):**

Address the following questions:
- Which approach works better overall?
- Does starting from the SFT checkpoint help or hinder performance?
- What are the failure modes of each approach?
- How did you choose the weighting for each reward component?

[Your discussion here]

---

## End of Submission